# Interpretable Credit Risk PD Scorecard & Cut-Off Optimization
**Author:** Shashank Singh (PGDM – Big Data Analytics, Goa Institute of Management)

This notebook builds a glass-box **300–850 Probability of Default (PD) Scorecard** using business-bucketed **Logistic Regression** and a **Depth-3 Decision Tree** on [All Lending Club Loan Data](https://www.kaggle.com/datasets/wordsforthewise/lending-club).


## 1. Imports & Load Application-Time Features (Zero Leakage)


In [ ]:
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import roc_auc_score, roc_curve

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

files = glob.glob("/kaggle/input/**/accepted_2007_to_2018*", recursive=True)
FILE_PATH = files[0] if files else "data/accepted_2007_to_2018Q4.csv.gz"
print("Loading:", FILE_PATH)

use_cols = [
    "issue_d", "loan_status", "loan_amnt", "term", "int_rate",
    "home_ownership", "annual_inc", "dti", "fico_range_low", "revol_util"
]
df = pd.read_csv(FILE_PATH, usecols=use_cols, low_memory=False)
df.shape


## 2. Target Definition & Chronological Out-of-Time (OOT) Split


In [ ]:
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])].copy()
df["target"] = (df["loan_status"] != "Fully Paid").astype(int)
df["issue_year"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce").dt.year
df = df.dropna(subset=["issue_year", "fico_range_low", "dti", "annual_inc", "loan_amnt"])

train = df[df["issue_year"].between(2013, 2016)].copy()
oot   = df[df["issue_year"].between(2017, 2018)].copy()

print(f"Total matured loans: {len(df):,} | Default Rate: {df['target'].mean():.2%}")
print(f"Train (2013-2016):   {len(train):,} | Default Rate: {train['target'].mean():.2%}")
print(f"OOT   (2017-2018):   {len(oot):,} | Default Rate: {oot['target'].mean():.2%}")


## 3. Business-Intuitive Bucketing & VIF Multicollinearity Check


In [ ]:
def bucket_features(data):
    d = pd.DataFrame(index=data.index)
    d["fico"] = pd.cut(data["fico_range_low"], bins=[0, 659, 699, 739, 900],
                       labels=["<660", "660-699", "700-739", "740+"])
    d["dti"]  = pd.cut(data["dti"], bins=[-1, 15, 25, 35, 1000],
                       labels=["0-15%", "15-25%", "25-35%", "35%+"])
    d["util"] = pd.cut(data["revol_util"].fillna(-1), bins=[-2, 0, 30, 60, 80, 1000],
                       labels=["Missing/0", "1-30%", "31-60%", "61-80%", "80%+"])
    d["lti"]  = pd.cut(data["loan_amnt"] / (data["annual_inc"] + 1.0),
                       bins=[-1, 0.15, 0.25, 0.35, 100],
                       labels=["<15%", "15-25%", "25-35%", "35%+"])
    d["term"] = data["term"].astype(str).str.strip()
    d["home"] = data["home_ownership"].where(
                    data["home_ownership"].isin(["MORTGAGE", "RENT", "OWN"]), "OTHER")
    return pd.get_dummies(d, drop_first=True, dtype=int)

X_train, y_train = bucket_features(train), train["target"].values
X_oot   = bucket_features(oot).reindex(columns=X_train.columns, fill_value=0)
y_oot   = oot["target"].values

# Compute VIF across all feature buckets
X_vals = X_train.astype(float).values
vif_rows = []
for i, col in enumerate(X_train.columns):
    r2 = LinearRegression().fit(np.delete(X_vals, i, axis=1), X_vals[:, i]).score(
        np.delete(X_vals, i, axis=1), X_vals[:, i])
    vif_rows.append({"Feature_Bucket": col, "VIF": round(1.0 / (1.0 - r2), 2)})
vif_df = pd.DataFrame(vif_rows).sort_values("VIF", ascending=False)
vif_df.to_csv(OUT_DIR / "vif_report.csv", index=False)
vif_df.head(10)


## 4. Fit Glass-Box Logistic Regression & Build Additive 300–850 Points Table


In [ ]:
lr = LogisticRegression(C=0.5, max_iter=1000)
lr.fit(X_train, y_train)

PDO, BASE_SCORE, BASE_ODDS = 20, 600, 50
factor = PDO / np.log(2.0)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)
base_points = int(round(offset - factor * lr.intercept_[0]))
bucket_points = np.round(-factor * lr.coef_[0]).astype(int)

points_table = pd.DataFrame({
    "Feature_Bucket": X_train.columns,
    "LogOdds_Coef": np.round(lr.coef_[0], 4),
    "Scorecard_Points": bucket_points,
}).sort_values("Scorecard_Points", ascending=False)
points_table.to_csv(OUT_DIR / "scorecard_points.csv", index=False)
print(f"Base Score: {base_points}")
points_table


## 5. OOT Banking Validation (ROC-AUC, Gini, KS-Statistic) & Charts


In [ ]:
pd_oot = np.clip(lr.predict_proba(X_oot)[:, 1], 1e-6, 1 - 1e-6)
auc = roc_auc_score(y_oot, pd_oot)
gini = 2 * auc - 1
fpr, tpr, _ = roc_curve(y_oot, pd_oot)
ks_idx = int(np.argmax(tpr - fpr))
ks = tpr[ks_idx] - fpr[ks_idx]
print(f"OOT ROC-AUC: {auc:.3f} | OOT Gini: {gini:.3f} | OOT KS: {ks:.1%}")

oot_scores = np.clip(base_points + X_oot.values.dot(bucket_points), 300, 850)
eval_df = pd.DataFrame({
    "score": oot_scores, "pd": pd_oot, "actual_default": y_oot, "ead": oot["loan_amnt"].values
})
eval_df["expected_loss"] = eval_df["pd"] * 0.75 * eval_df["ead"]

# Plot KS Curve
x_axis = np.linspace(0, 1, len(tpr))
plt.figure(figsize=(7.5, 4.5))
plt.plot(x_axis, tpr, label="Cumulative Defaults (Bad)", color="#C0392B", lw=2)
plt.plot(x_axis, fpr, label="Cumulative Paid (Good)", color="#1F618D", lw=2)
plt.vlines(x_axis[ks_idx], fpr[ks_idx], tpr[ks_idx], colors="#27AE60", linestyles="--", label=f"KS = {ks:.1%}")
plt.title("Out-of-Time KS Curve")
plt.xlabel("Population Ranked by Risk"); plt.ylabel("Cumulative Proportion")
plt.legend(frameon=False); plt.grid(alpha=0.25); plt.tight_layout()
plt.savefig(OUT_DIR / "ks_curve.png", dpi=180); plt.show()


## 6. Expected Loss Cut-Off Simulation, Decline Reason Codes & Depth-3 Policy Tree


In [ ]:
rows = []
for rej_pct in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]:
    cutoff = eval_df["score"].quantile(rej_pct)
    rej = eval_df[eval_df["score"] < cutoff]
    app = eval_df[eval_df["score"] >= cutoff]
    rows.append({
        "Reject_Rate": rej_pct,
        "Approval_Rate": 1 - rej_pct,
        "Score_Cutoff": round(float(cutoff), 0),
        "Approved_Default_Rate": app["actual_default"].mean(),
        "Expected_Loss_Avoided": rej["expected_loss"].sum() / eval_df["expected_loss"].sum(),
    })
cutoff_df = pd.DataFrame(rows)
cutoff_df.to_csv(OUT_DIR / "cutoff_simulation.csv", index=False)
display(cutoff_df)

# Sample Decline Reason Codes
worst_pos = int(eval_df["score"].idxmin())
active = X_oot.iloc[worst_pos]
reasons = points_table[points_table["Feature_Bucket"].isin(active[active == 1].index)]
print(f"Sample Declined Applicant Score: {eval_df.loc[worst_pos, 'score']:.0f}")
display(reasons.sort_values("Scorecard_Points").head(3))

# Companion Depth-3 Decision Tree Policy Rules
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=500, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)
print(export_text(tree, feature_names=list(X_train.columns)))
